# 01 · First look at negative prices

First exploration of the interim tables: what negative prices look like, when they happen, and how well AEMO's own day-ahead forecast catches them.

- **Input:** `data/interim/` (built by `make data`)
- **Time:** NEM time (UTC+10, no daylight saving); timestamps mark the **end** of each interval

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from negprice.config import INTERIM_DATA_DIR

price = pd.read_parquet(INTERIM_DATA_DIR / "dispatchprice.parquet")        # actual price, every 5 min
demand = pd.read_parquet(INTERIM_DATA_DIR / "dispatchregionsum.parquet")   # actual demand, every 5 min
pv = pd.read_parquet(INTERIM_DATA_DIR / "rooftop_pv_actual.parquet")       # rooftop solar, every 30 min
pv = pv[pv["TYPE"] == "MEASUREMENT"]
price.head()

## 1. One week in SA

What a typical spring week looks like: prices fall below zero around midday, when rooftop solar pushes demand on the grid down.

In [ ]:
start, end = "2025-10-06", "2025-10-13"
sa = price[(price["REGIONID"] == "SA1") & (price["SETTLEMENTDATE"] >= start) & (price["SETTLEMENTDATE"] < end)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sa["SETTLEMENTDATE"], sa["RRP"], lw=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylim(-200, 400)
ax.set_ylabel("Price ($/MWh)")
ax.set_title("SA price, one week in October 2025");

In [ ]:
d = demand[(demand["REGIONID"] == "SA1") & (demand["SETTLEMENTDATE"] >= start) & (demand["SETTLEMENTDATE"] < end)]
p = pv[(pv["REGIONID"] == "SA1") & (pv["INTERVAL_DATETIME"] >= start) & (pv["INTERVAL_DATETIME"] < end)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(d["SETTLEMENTDATE"], d["TOTALDEMAND"], label="Demand on the grid")
ax.plot(p["INTERVAL_DATETIME"], p["POWER"], label="Rooftop solar")
ax.set_ylabel("MW")
ax.set_title("SA demand and rooftop solar, same week")
ax.legend();

## 2. When are prices negative?

Share of 5-minute intervals with a negative price, by the hour the interval starts in.

In [ ]:
price["hour"] = (price["SETTLEMENTDATE"] - pd.Timedelta("5min")).dt.hour   # hour the interval starts in
price["negative"] = price["RRP"] < 0

share = price.groupby(["hour", "REGIONID"])["negative"].mean().unstack() * 100
share.plot(figsize=(10, 4), ylabel="% of intervals negative", title="Negative prices by hour, Oct 2021 – Sep 2026");

In [ ]:
price["year"] = price["SETTLEMENTDATE"].dt.year
(price.groupby(["year", "REGIONID"])["negative"].mean().unstack() * 100).round(1)

**Findings**

- Negative prices are a **midday solar** phenomenon: peak at 11:00–13:00 (SA ~58%, VIC ~49%, NSW ~25% of intervals). SA and VIC also see ~10–15% overnight (wind); NSW almost never goes negative at night.
- By year: SA 19–30%, VIC 14–24%, NSW 3–14% of intervals. NSW jumped to 14% in 2025.
- 2021 covers Oct–Dec only and 2026 Jan–Sep only, so those two years aren't comparable with full years (seasonality).

## 3. AEMO's day-ahead forecast (predispatch, 13:00 run on D-1)

The benchmark to beat. Actual 5-minute prices are averaged to half-hours (interval-end labels: the half-hour ending 12:30 holds the intervals ending 12:05 … 12:30) and joined to the forecast.

In [ ]:
fc = pd.read_parquet(INTERIM_DATA_DIR / "predispatch_price.parquet")
fc.head()

In [ ]:
price["DATETIME"] = price["SETTLEMENTDATE"].dt.ceil("30min")
actual = price.groupby(["REGIONID", "DATETIME"], as_index=False)["RRP"].mean()

both = actual.merge(fc, on=["REGIONID", "DATETIME"], suffixes=("_actual", "_forecast"))
print(len(actual), "actual half-hours |", len(both), "with a forecast")
both.head()

The 4,464 half-hours without a forecast are AEMO's predispatch archive gap (target days 2 Oct – 1 Nov 2022).

In [ ]:
day = pd.Timestamp("2025-10-08")
one = both[(both["REGIONID"] == "NSW1") & (both["DATETIME"] > day) & (both["DATETIME"] <= day + pd.Timedelta("1D"))]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(one["DATETIME"], one["RRP_actual"], marker="o", ms=3, label="Actual")
ax.plot(one["DATETIME"], one["RRP_forecast"], marker="o", ms=3, label="AEMO forecast (made 13:00 the day before)")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Price ($/MWh)")
ax.set_title("NSW, 8 Oct 2025: actual vs AEMO forecast")
ax.legend();

In [ ]:
both["neg_actual"] = both["RRP_actual"] < 0
both["neg_forecast"] = both["RRP_forecast"] < 0

for region, g in both.groupby("REGIONID"):
    table = pd.crosstab(g["neg_forecast"], g["neg_actual"])
    recall = table.loc[True, True] / table[True].sum() * 100
    precision = table.loc[True, True] / table.loc[True].sum() * 100
    print(region)
    print(table)
    print(f"Caught {recall:.0f}% of actual negative half-hours (recall); "
          f"right {precision:.0f}% of the times it said negative (precision)\n")

In [ ]:
both["hour"] = (both["DATETIME"] - pd.Timedelta("30min")).dt.hour   # hour the half-hour starts in

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (region, g) in zip(axes, both.groupby("REGIONID")):
    (g.groupby("hour")[["neg_actual", "neg_forecast"]].mean() * 100).plot(ax=ax, title=region)
axes[0].set_ylabel("% of half-hours negative")
fig.suptitle("Negative half-hours: actual vs AEMO forecast, by hour");

**Findings**

| Region | Recall | Precision |
|---|---|---|
| NSW | 80% | 50% |
| SA | 77% | 84% |
| VIC | 75% | 80% |

- AEMO's forecast is a strong baseline in SA and VIC.
- In NSW it **over-forecasts** negative prices: at midday it predicts ~39% negative half-hours against ~25% actual, so half of its "negative" calls are wrong.